<a href="https://colab.research.google.com/github/rishabhucb/court-keypoints/blob/main/day4_homography_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q ultralytics
from google.colab import drive
drive.mount('/content/drive')

import cv2, numpy as np, glob, os, random
import matplotlib.pyplot as plt
from ultralytics import YOLO

PROJECT_DIR = '/content/drive/MyDrive/court_keypoints'
TEST_IMGS = sorted(glob.glob(f'{PROJECT_DIR}/court_v4/test/images/*'))
FIG_DIR = f'{PROJECT_DIR}/figures'
os.makedirs(FIG_DIR, exist_ok=True)

kp_model = YOLO(f'{PROJECT_DIR}/runs/baseline_v4/weights/last.pt')   # your court model
person_model = YOLO('yolo11n.pt')                                    # standard model that detects people
print(len(TEST_IMGS), "test images")

In [ ]:
COURT_FT = {
    0: (0, 0),      1: (50, 0),      2: (0, 42),     12: (50, 42),   # corners / half-court sideline
    3: (19, 0),     4: (31, 0),      5: (31, 19),    6: (19, 19),    # paint (12 ft wide), FT line at 19 ft
    7: (5.25, 0),   8: (5.25, 5.25), 10: (44.75, 5.25), 11: (44.75, 0),  # 3-pt line ends
    9: (25, 25),                                                     # top of the arc (19'9" from hoop)
}
S, M = 10, 20                                   # 10 pixels per foot, 20 px margin
W, H = int(50*S + 2*M), int(42*S + 2*M)
P = lambda x, y: (int(M + x*S), int(M + y*S))
COURT_PX = {k: (M + x*S, M + y*S) for k, (x, y) in COURT_FT.items()}

def draw_court(img=None, color=(255, 255, 255)):
    if img is None:
        img = np.full((H, W, 3), (80, 140, 190), np.uint8)          # wood color
    cv2.rectangle(img, P(0, 0), P(50, 42), color, 2)                 # outline + half-court line
    cv2.rectangle(img, P(19, 0), P(31, 19), color, 2)                # paint
    cv2.circle(img, P(25, 19), 6*S, color, 2)                        # free-throw circle
    cv2.line(img, P(5.25, 0), P(5.25, 5.25), color, 2)               # corner-3 lines
    cv2.line(img, P(44.75, 0), P(44.75, 5.25), color, 2)
    cv2.ellipse(img, P(25, 5.25), (int(19.75*S),)*2, 0, 0, 180, color, 2)   # 3-pt arc
    cv2.circle(img, P(25, 5.25), int(0.75*S), color, 2)              # hoop
    cv2.ellipse(img, P(25, 42), (6*S, 6*S), 0, 180, 360, color, 2)   # center circle
    return img

plt.imshow(cv2.cvtColor(draw_court(), cv2.COLOR_BGR2RGB)); plt.axis('off'); plt.show()

In [ ]:
for i in [7, 8, 9, 10, 11]:                       # use the measured 3-point positions
    COURT_FT[i] = measured[i][:2]
COURT_PX = {k: (M + x*S, M + y*S) for k, (x, y) in COURT_FT.items()}
USE_IDS = [3, 4, 5, 6, 7, 8, 9, 10, 11]           # skip the corner/sideline points
print({i: tuple(round(c, 1) for c in COURT_FT[i]) for i in USE_IDS})

In [ ]:
def court_map(img_path, conf_th=0.5, name=None):
    img = cv2.imread(img_path)
    r = kp_model.predict(img, conf=0.25, verbose=False)[0]
    if r.keypoints is None or len(r.boxes) == 0:
        print("No court found"); return None
    best = int(r.boxes.conf.argmax())
    xy = r.keypoints.xy[best].cpu().numpy()
    kc = r.keypoints.conf[best].cpu().numpy()
    ids = [i for i in USE_IDS if kc[i] > conf_th]
    if len(ids) < 4:
        print("Only", len(ids), "confident points; need 4"); return None

    # Homography: image pixels -> court diagram pixels
    Hm, _ = cv2.findHomography(np.float32([xy[i] for i in ids]),
                               np.float32([COURT_PX[i] for i in ids]), cv2.RANSAC, 15.0)

    top = draw_court(cv2.warpPerspective(img, Hm, (W, H)))         # bird's-eye view

    # Players: feet = bottom-center of each person box
    boxes = person_model.predict(img, classes=[0], conf=0.4, verbose=False)[0].boxes.xyxy.cpu().numpy()
    court2d = draw_court()
    if len(boxes):
        feet = np.float32([[(b[0]+b[2])/2, b[3]] for b in boxes]).reshape(-1, 1, 2)
        for x, y in cv2.perspectiveTransform(feet, Hm).reshape(-1, 2):
            if 0 <= x < W and 0 <= y < H:
                cv2.circle(court2d, (int(x), int(y)), 8, (0, 0, 255), -1)

    vis = img.copy()
    for b in boxes:
        cv2.rectangle(vis, (int(b[0]), int(b[1])), (int(b[2]), int(b[3])), (0, 0, 255), 2)
    for i in ids:
        cv2.circle(vis, tuple(map(int, xy[i])), 7, (0, 255, 255), -1)
        cv2.putText(vis, str(i), (int(xy[i][0])+8, int(xy[i][1])-8), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)

    fig, ax = plt.subplots(1, 3, figsize=(22, 6))
    for a, im, t in zip(ax, [vis, top, court2d],
                        ['1. Court keypoints + players', '2. Top-down warp', '3. Players on 2D court']):
        a.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB)); a.set_title(t, fontsize=14); a.axis('off')
    if name:
        plt.savefig(f'{FIG_DIR}/{name}.png', bbox_inches='tight')
    plt.show()
    print(f"Used {len(ids)} keypoints: {ids}")
    return Hm

for k, p in enumerate(random.sample(TEST_IMGS, 3)):
    court_map(p, name=f'court_map_{k}')

In [ ]:
def error_feet(img_path, Hm):
    h, w = cv2.imread(img_path).shape[:2]
    lbl = img_path.replace('/images/', '/labels/').rsplit('.', 1)[0] + '.txt'
    v = list(map(float, open(lbl).readline().split()))[5:]
    errs = []
    for i in USE_IDS:
        if v[3*i+2] > 0:
            p = cv2.perspectiveTransform(np.float32([[[v[3*i]*w, v[3*i+1]*h]]]), Hm)[0, 0]
            fx, fy = (p[0]-M)/S, (p[1]-M)/S
            errs.append(np.hypot(fx - COURT_FT[i][0], fy - COURT_FT[i][1]))
    return np.mean(errs) if errs else None

all_errs = []
for p in TEST_IMGS:
    Hm = court_map(p) if False else None   # (no plots here)
    img = cv2.imread(p)
    r = kp_model.predict(img, conf=0.25, verbose=False)[0]
    if r.keypoints is None or len(r.boxes) == 0: continue
    b = int(r.boxes.conf.argmax())
    xy, kc = r.keypoints.xy[b].cpu().numpy(), r.keypoints.conf[b].cpu().numpy()
    ids = [i for i in USE_IDS if kc[i] > 0.5]
    if len(ids) < 4: continue
    Hm, _ = cv2.findHomography(np.float32([xy[i] for i in ids]),
                               np.float32([COURT_PX[i] for i in ids]), cv2.RANSAC, 15.0)
    e = error_feet(p, Hm)
    if e is not None: all_errs.append(e)

print(f"Images evaluated: {len(all_errs)} / {len(TEST_IMGS)}")
print(f"Mean court-mapping error: {np.mean(all_errs):.2f} ft  (median {np.median(all_errs):.2f} ft)")

In [ ]:
PAINT = [3, 4, 5, 6]
proj = {i: [] for i in range(13)}
for lbl in glob.glob(f'{PROJECT_DIR}/court_v4/train/labels/*.txt'):
    v = list(map(float, open(lbl).readline().split()))[5:]
    pts = {i: (v[3*i], v[3*i+1]) for i in range(13) if v[3*i+2] > 0}
    if not all(i in pts for i in PAINT):
        continue
    Hgt = cv2.getPerspectiveTransform(np.float32([pts[i] for i in PAINT]),
                                      np.float32([COURT_FT[i] for i in PAINT]))
    for i, (x, y) in pts.items():
        X, Y = cv2.perspectiveTransform(np.float32([[[x, y]]]), Hgt)[0, 0]
        proj[i].append((X, Y))

measured = {}
for i in range(13):
    if proj[i]:
        a = np.array(proj[i]); med = np.median(a, 0)
        spread = np.median(np.hypot(*(a - med).T))
        measured[i] = (float(med[0]), float(med[1]), float(spread))
        print(f"ID {i:2d}: measured ({med[0]:6.1f}, {med[1]:6.1f}) ft | assumed {COURT_FT[i]} | spread {spread:5.1f} ft | n={len(a)}")

In [ ]:
def opencv_lines(img_path, name=None):
    img = cv2.imread(img_path)
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv, (0, 0, 170), (180, 60, 255))            # bright/white pixels
    edges = cv2.Canny(cv2.GaussianBlur(mask, (5, 5), 0), 50, 150)
    lines = cv2.HoughLinesP(edges, 1, np.pi/180, 80, minLineLength=60, maxLineGap=15)
    out = img.copy()
    for x1, y1, x2, y2 in (lines[:, 0] if lines is not None else []):
        cv2.line(out, (x1, y1), (x2, y2), (0, 0, 255), 2)
    fig, ax = plt.subplots(1, 2, figsize=(16, 5))
    ax[0].imshow(mask, cmap='gray'); ax[0].set_title('White-pixel mask')
    ax[1].imshow(cv2.cvtColor(out, cv2.COLOR_BGR2RGB)); ax[1].set_title(f'Hough lines: {0 if lines is None else len(lines)}')
    for a in ax: a.axis('off')
    if name: plt.savefig(f'{FIG_DIR}/{name}.png', bbox_inches='tight')
    plt.show()

opencv_lines(TEST_IMGS[0], name='opencv_lines')

In [ ]:
print(USE_IDS)
print(COURT_FT[8], COURT_FT[9])

In [ ]:
# =====================================================================
# 3-POINT LINE + 2PT / 3PT CLASSIFICATION (complete, self-contained)
# Needs from earlier cells: kp_model, person_model, TEST_IMGS, FIG_DIR,
#                           draw_court, S, M
# =====================================================================
import numpy as np, cv2, random
import matplotlib.pyplot as plt

# ---- 1. Court positions (ft) of the reliable keypoints, measured from the labels on Day 4
COURT_FT = {3: (19, 0), 4: (31, 0), 5: (31, 19), 6: (19, 19),
            7: (6.8, 1.4), 8: (9.1, 16.8), 9: (24.5, 25.2), 10: (39.9, 19.7), 11: (47.7, 6.6)}
COURT_PX = {k: (M + x * S, M + y * S) for k, (x, y) in COURT_FT.items()}
USE_IDS = [3, 4, 5, 6, 7, 8, 9, 10, 11]
ARC_IDS = [7, 8, 9, 10, 11]          # the 5 points that lie on the 3-point line
MARGIN = 1.5                         # ft; closer than this to the line = "CLOSE" (our mapping error)

# ---- 2. Fit the 3-point arc (circle) through the 5 arc points, in court feet
pts = np.array([COURT_FT[i] for i in ARC_IDS], float)
A = np.c_[2 * pts, np.ones(len(pts))]
cx, cy, c = np.linalg.lstsq(A, (pts ** 2).sum(1), rcond=None)[0]
R_fit = float(np.sqrt(c + cx ** 2 + cy ** 2))
a_start = np.arctan2(COURT_FT[7][1] - cy, COURT_FT[7][0] - cx)
a_end = np.arctan2(COURT_FT[11][1] - cy, COURT_FT[11][0] - cx)
if a_start < 0:
    a_start += 2 * np.pi                  # go over the top of the arc, not underneath
print(f"3-point arc: center ({cx:.2f}, {cy:.2f}) ft, radius {R_fit:.2f} ft")

def three_pt_line_ft(n=200):
    """3-point line in court feet, from end point 7 over the top to end point 11."""
    a = np.linspace(a_start, a_end, n)
    return np.stack([cx + R_fit * np.cos(a), cy + R_fit * np.sin(a)], 1).astype(np.float32)

def shot_zone(x, y):
    """Court position (ft) -> '2PT', '3PT' or 'CLOSE'."""
    d = np.hypot(x - cx, y - cy) - R_fit
    if abs(d) < MARGIN:
        return 'CLOSE'
    return '3PT' if d > 0 else '2PT'

# ---- 3. Keypoints and homography for one image
def detect_keypoints(img):
    r = kp_model.predict(img, conf=0.25, verbose=False)[0]
    if r.keypoints is None or len(r.boxes) == 0:
        return None, None
    b = int(r.boxes.conf.argmax())
    return r.keypoints.xy[b].cpu().numpy(), r.keypoints.conf[b].cpu().numpy()

def get_H(xy, kc, conf_th=0.5):
    ids = [i for i in USE_IDS if kc[i] > conf_th]
    if len(ids) < 4:
        return None
    Hm, _ = cv2.findHomography(np.float32([xy[i] for i in ids]),
                               np.float32([COURT_PX[i] for i in ids]), cv2.RANSAC, 15.0)
    return Hm

# ---- 4. Draw the line straight through the 5 detected arc points (most accurate on the image)
def three_pt_curve_image(xy, kc, conf_th=0.5):
    if not all(kc[i] > conf_th for i in ARC_IDS):
        return None
    (ex, ey), (d1, d2), ang = cv2.fitEllipse(np.float32([xy[i] for i in ARC_IDS]))
    poly = cv2.ellipse2Poly((int(ex), int(ey)), (int(d1 / 2), int(d2 / 2)), int(ang), 0, 360, 1)
    near = lambda p: int(np.argmin(np.hypot(*(poly - p).T)))
    i7, i9, i11, n = near(xy[7]), near(xy[9]), near(xy[11]), len(poly)
    path = [(i7 + k) % n for k in range((i11 - i7) % n + 1)]
    if i9 not in path:
        path = [(i11 + k) % n for k in range((i7 - i11) % n + 1)]
    return poly[path].reshape(-1, 1, 2)

# ---- 5. Full demo for one image
ZONE_COLOR = {'2PT': (214, 120, 42), '3PT': (52, 104, 235), 'CLOSE': (160, 160, 160)}  # blue, orange, gray
LINE_COLOR = (0, 215, 255)                                                             # yellow

def three_point_demo(img_path, name=None):
    img = cv2.imread(img_path)
    xy, kc = detect_keypoints(img)
    if xy is None:
        print("Court not found"); return
    Hm = get_H(xy, kc)
    if Hm is None:
        print("Not enough keypoints"); return
    Hinv = np.linalg.inv(Hm)

    # 3-point line: direct fit through the 5 points if all are visible, otherwise via homography
    line_px = three_pt_line_ft() * S + M
    line_img = three_pt_curve_image(xy, kc)
    method = "direct fit through 5 points"
    if line_img is None:
        line_img = cv2.perspectiveTransform(line_px.reshape(-1, 1, 2), Hinv)
        method = "homography"

    vis, court2d = img.copy(), draw_court()
    cv2.polylines(vis, [line_img.astype(np.int32)], False, LINE_COLOR, 3)
    cv2.polylines(court2d, [line_px.astype(np.int32).reshape(-1, 1, 2)], False, LINE_COLOR, 3)
    for i in ARC_IDS:                                          # show the 5 arc keypoints
        cv2.circle(vis, tuple(map(int, xy[i])), 6, LINE_COLOR, -1)

    # Players: feet -> court feet -> 2PT / 3PT / CLOSE
    boxes = person_model.predict(img, classes=[0], conf=0.4, verbose=False)[0].boxes.xyxy.cpu().numpy()
    counts = {'2PT': 0, '3PT': 0, 'CLOSE': 0}
    for b in boxes:
        foot = np.float32([[[(b[0] + b[2]) / 2, b[3]]]])
        px, py = cv2.perspectiveTransform(foot, Hm)[0, 0]
        fx, fy = (px - M) / S, (py - M) / S
        if not (0 <= fx <= 50 and 0 <= fy <= 42):
            continue                                           # off the court (bench, fans)
        z = shot_zone(fx, fy)
        counts[z] += 1
        col = ZONE_COLOR[z]
        cv2.rectangle(vis, (int(b[0]), int(b[1])), (int(b[2]), int(b[3])), col, 3)
        cv2.putText(vis, z, (int(b[0]), int(b[1]) - 8), cv2.FONT_HERSHEY_SIMPLEX, 0.8, col, 2)
        cv2.circle(court2d, (int(px), int(py)), 9, col, -1)
        cv2.putText(court2d, z, (int(px) + 10, int(py) + 5), cv2.FONT_HERSHEY_SIMPLEX, 0.45, col, 1)

    fig, ax = plt.subplots(1, 2, figsize=(20, 6), gridspec_kw={'width_ratios': [1.6, 1]})
    ax[0].imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    ax[0].set_title(f'3-point line on camera image (yellow, {method})', fontsize=13)
    ax[1].imshow(cv2.cvtColor(court2d, cv2.COLOR_BGR2RGB))
    ax[1].set_title(f"{counts['2PT']} inside (2PT, blue) | {counts['3PT']} outside (3PT, orange) | "
                    f"{counts['CLOSE']} on the line (gray)", fontsize=13)
    for a in ax:
        a.axis('off')
    if name:
        plt.savefig(f'{FIG_DIR}/{name}.png', bbox_inches='tight')
    plt.show()

# ---- 6. Run on 3 test images (saved to Drive as three_point_0/1/2.png)
for k, p in enumerate(random.sample(TEST_IMGS, 3)):
    three_point_demo(p, name=f'three_point_{k}')

In [ ]:
# Fit a circle through the 3-point keypoints (wing, top, wing) in court feet
pts = np.array([COURT_FT[i] for i in [8, 9, 10]], float)
A = np.c_[2 * pts, np.ones(len(pts))]
cx, cy, c = np.linalg.lstsq(A, (pts ** 2).sum(1), rcond=None)[0]
R_fit = float(np.sqrt(c + cx ** 2 + cy ** 2))

print(f"Official rule : hoop (25.0, 5.25), radius 19.75 ft")
print(f"Fitted to data: hoop ({cx:.2f}, {cy:.2f}), radius {R_fit:.2f} ft")

HOOP, R3 = (float(cx), float(cy)), R_fit      # use the fitted arc from now on

In [ ]:
def line_from(hx, hy, r, n=200):
    left  = [(hx - r, 0), (hx - r, hy)]
    arc   = [(hx + r*np.cos(a), hy + r*np.sin(a)) for a in np.linspace(np.pi, 0, n)]
    right = [(hx + r, hy), (hx + r, 0)]
    return np.array(left + arc + right, np.float32)

def compare_lines(img_path, candidates):
    img = cv2.imread(img_path)
    Hinv = np.linalg.inv(get_H(img))
    colors = [(0, 215, 255), (255, 0, 255), (0, 255, 0)]          # yellow, magenta, green
    for (hx, hy, r), col in zip(candidates, colors):
        px = line_from(hx, hy, r) * S + M
        im = cv2.perspectiveTransform(px.reshape(-1, 1, 2), Hinv)
        cv2.polylines(img, [im.astype(np.int32)], False, col, 2)
    plt.figure(figsize=(16, 9)); plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); plt.axis('off')
    plt.title('Yellow = official rule | Magenta = fitted | Green = fitted + 0.5 ft'); plt.show()

compare_lines(TEST_IMGS[0], [(25, 5.25, 19.75), (HOOP[0], HOOP[1], R3), (HOOP[0], HOOP[1], R3 + 0.5)])

In [ ]:
# Fit the arc through ALL five 3-point keypoints: ends (7, 11), wings (8, 10), top (9)
ids3 = [7, 8, 9, 10, 11]
pts = np.array([COURT_FT[i] for i in ids3], float)
A = np.c_[2 * pts, np.ones(len(pts))]
cx, cy, c = np.linalg.lstsq(A, (pts ** 2).sum(1), rcond=None)[0]
R_fit = float(np.sqrt(c + cx ** 2 + cy ** 2))
HOOP, R3 = (float(cx), float(cy)), R_fit

# Angles of the two end points, so the line starts at 7 and stops at 11
a_start = np.arctan2(COURT_FT[7][1] - cy, COURT_FT[7][0] - cx)
a_end   = np.arctan2(COURT_FT[11][1] - cy, COURT_FT[11][0] - cx)
if a_start < 0:
    a_start += 2 * np.pi          # go over the top of the arc, not underneath

print(f"Fitted arc: center ({cx:.2f}, {cy:.2f}) ft, radius {R_fit:.2f} ft")
for i in ids3:
    d = np.hypot(COURT_FT[i][0] - cx, COURT_FT[i][1] - cy) - R_fit
    print(f"  ID {i:2d}: {d:+.2f} ft from the fitted line")

# Replace the line and the 2PT/3PT rule with the fitted version
def three_pt_line_ft(n=200):
    a = np.linspace(a_start, a_end, n)
    return np.stack([cx + R_fit * np.cos(a), cy + R_fit * np.sin(a)], 1).astype(np.float32)

def shot_zone(x, y):
    d = np.hypot(x - cx, y - cy) - R_fit
    if abs(d) < MARGIN:
        return 'CLOSE'
    return '3PT' if d > 0 else '2PT'

In [ ]:
def three_pt_curve_image(img, conf_th=0.5):
    """3-point line drawn directly through the model's 5 detected arc points (image pixels)."""
    r = kp_model.predict(img, conf=0.25, verbose=False)[0]
    if r.keypoints is None or len(r.boxes) == 0:
        return None
    b = int(r.boxes.conf.argmax())
    xy, kc = r.keypoints.xy[b].cpu().numpy(), r.keypoints.conf[b].cpu().numpy()
    if not all(kc[i] > conf_th for i in [7, 8, 9, 10, 11]):
        return None                                     # need all 5 points; otherwise fall back
    (ex, ey), (d1, d2), ang = cv2.fitEllipse(np.float32([xy[i] for i in [7, 8, 9, 10, 11]]))
    poly = cv2.ellipse2Poly((int(ex), int(ey)), (int(d1 / 2), int(d2 / 2)), int(ang), 0, 360, 1)
    near = lambda p: int(np.argmin(np.hypot(*(poly - p).T)))
    i7, i9, i11, n = near(xy[7]), near(xy[9]), near(xy[11]), len(poly)
    path = [(i7 + k) % n for k in range((i11 - i7) % n + 1)]      # go from end 7 to end 11 ...
    if i9 not in path:                                             # ... the way that passes the top (9)
        path = [(i11 + k) % n for k in range((i7 - i11) % n + 1)]
    return poly[path]